# mermaidx in Jupyter

`mermaidx` renders Mermaid diagrams without a browser, Node.js, or npm — and that includes right here in the notebook. This walks through everything: automatic inline display, PNG/PDF/ASCII export, themes, and the optional "live" mode that renders with the real mermaid.js in your browser.

```
pip install mermaidx
```


In [ ]:
import mermaidx

mermaidx.__version__

## Automatic display

A `Diagram` shows up inline automatically when it's the last expression in a cell — no `.show()`, no `plt.figure()` boilerplate, just like a DataFrame or a matplotlib figure. This works because `Diagram` implements IPython's `_repr_svg_()` rich-display hook, and renders to a real, static SVG the first time it's needed (then caches it).

In [ ]:
d = mermaidx.render("""
flowchart TD
    A[Start] --> B{Is it working?}
    B -->|Yes| C[Great!]
    B -->|No| D[Debug it]
    D --> E[Try again]
    E --> A
""")

d

## `.show()`

Useful when the diagram isn't the last line in the cell — e.g. inside a loop, or after some other output. Displays exactly what `.svg()` returns, so what you see is always this package's own render pipeline output -- not a separate re-render through some other engine.

In [ ]:
flowchart = "graph LR; A-->B{ok?}; B-->|yes|C[Done]; B-->|no|D[Retry]"

for theme in ["default", "forest", "dark", "neutral"]:
    print(theme)
    mermaidx.render(flowchart, theme=theme).show()

## PNG

In [ ]:
png_bytes = d.png(width=500, background="#ffffff")

from IPython.display import Image
Image(png_bytes)

## PDF, raw pixels, numpy, ASCII

All lazy, all cached — calling `.svg()` again after any of these doesn't re-render.

In [ ]:
pdf_bytes = d.pdf(pdf_format="A4", pdf_margin="1cm")
print(f"PDF: {len(pdf_bytes):,} bytes")

raw, w, h = d.raw()
print(f"raw RGBA: {w}x{h}, {len(raw):,} bytes")

arr = d.numpy()  # needs numpy installed
print(f"numpy: {arr.shape} {arr.dtype}")

print(d.ascii())

## Saving to disk

Format is inferred from the extension, or forced with `format=`.

In [ ]:
d.save("diagram.svg")
d.save("diagram.png", width=1200)
d.save("diagram.pdf", pdf_format="A4")
d.save("diagram.txt")                     # ASCII, from the .txt extension
d.save("diagram.backup", format="png")   # forced, ignoring the odd extension

## Backends

`'js'` (mermaidx's own engine) is always available. If the optional [`mmdr`](https://github.com/mohammadraziei/mmdr) package is installed, its backends show up too, and PDF/raw/numpy work for them the same way — mermaidx supplies those from its own resvg pipeline even for backends that don't natively support them.

In [ ]:
mermaidx.backends()

## Batch rendering

CPU-bound work parallelizes with real processes, not `async` — each worker starts its own engine once and reuses it for every diagram routed to it.

In [ ]:
sources = [
    "graph LR; A-->B",
    "sequenceDiagram\n  Alice->>Bob: Hi",
    "stateDiagram-v2\n  [*] --> Idle\n  Idle --> Running",
]

diagrams = mermaidx.render_many(sources, workers=2)
diagrams[1]